# Nib v2.x — Faithful Rewrite LoRA (Colab Free Tier)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abgnydn/quill/blob/master/train/colab/train_nib_v2.ipynb)

Train a LoRA on Qwen 2.5-1.5B-Instruct using rejection-sampling-filtered samples from the local eval pipeline, then export either:

- **Merged GGUF** (~940 MB) — drop-in replacement for the base, no adapter plumbing needed.
- **Adapter-only GGUF** (~36 MB) — layers on top of stock Qwen at runtime; cheap to iterate on.

**Runtime:** T4 GPU (free). **Wall clock:** ~10–15 min training + ~5 min export.

**Dataset:** auto-pulled from this repo on GitHub. Change `DATA_FILE` in Cell 2 to switch between `rsft-round1.jsonl` (133 samples, v2.0) and `rsft-round2.jsonl` (551 samples, v2.1).

## 1. Install (Unsloth + GGUF tooling)

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Need TRL >=0.12 for SFTConfig + processing_class API.
# Unsloth's deps pin an older TRL; explicit upgrade overrides that.
!pip install -q -U "trl>=0.12,<0.20" peft accelerate bitsandbytes


## 2. Fetch the RSFT dataset from GitHub

No upload needed — the dataset lives in this repo (`train/data/`) and gets `curl`'d on demand. Edit `DATA_FILE` to pick which round to train on.

In [ ]:
# Pull the RSFT dataset straight from the repo — no manual upload needed.
# Swap DATA_FILE between rounds:
#   rsft-round1.jsonl  — 133 samples (v2.0 — sampled against LFM2.5-1.2B)
#   rsft-round2.jsonl  — 551 samples (v2.1 — sampled against Qwen 2.5 base)
import os
DATA_FILE = "rsft-round2.jsonl"
DATA_URL = f"https://raw.githubusercontent.com/abgnydn/quill/master/train/data/{DATA_FILE}"
if not os.path.exists(DATA_FILE):
    !curl -sL {DATA_URL} -o {DATA_FILE}
DATA_PATH = DATA_FILE
!wc -l {DATA_PATH}

## 3. Load base model + attach LoRA

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024  # our samples are tiny; no need for more
BASE = "unsloth/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto: bf16 on A100/L4, fp16 on T4
    load_in_4bit=True,     # required for T4 to fit comfortably
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
)
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Format dataset (ChatML, completion-only loss)

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=DATA_PATH, split="train")
print("raw rows:", len(raw))
print("sample:", raw[0])

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = raw.map(format_example, remove_columns=raw.column_names)
print("\nformatted preview:")
print(dataset[0]["text"])

## 5. Train

In [ ]:
import os
# Unsloth 2024.11+ skips materializing logits by default for speed, but
# DataCollatorForCompletionOnlyLM needs them to compute the masked loss.
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

# Mask everything before the assistant turn so we only learn the
# rewrite output, not the system/user echo.
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    tokenizer=tokenizer,
)

# TRL 1.x renames: tokenizer= -> processing_class=, max_seq_length -> max_length,
# and dataset_text_field / packing move from SFTTrainer kwargs into SFTConfig.
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=collator,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",
    ),
)

stats = trainer.train()
print(stats)


## 6. Sanity inference (3 cases from the eval set)

In [ ]:
FastLanguageModel.for_inference(model)

SANITY = [
    "I has a apple. He don't likes it.",
    "manage users: 1 Day",
    "The api endpoint /v2/search returned a 500 error after $47.30 in usage.",
]

DEFAULT_INSTR = (
    "You are a copy editor. Fix the grammar and improve clarity. "
    "Output only the corrected text, nothing else."
)

for src in SANITY:
    msgs = [
        {"role": "system", "content": DEFAULT_INSTR},
        {"role": "user",   "content": src},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"SRC: {src}")
    print(f"OUT: {text.strip()}\n")

## 7. Merge LoRA into base + export Q4_K_M GGUF

Merged model goes out as a standalone GGUF — drop-in replacement for the current `lfm2.5-1.2b-instruct-q4_k_m.gguf` Nib bundles. No `--adapter` support needed in `quill-rewrite`.

In [ ]:
# Unsloth handles: merge LoRA → save HF format → clone llama.cpp →
# convert to GGUF → quantize to Q4_K_M. ~3-4 min on T4.
model.save_pretrained_gguf(
    "nib-qwen-q4_k_m",
    tokenizer,
    quantization_method="q4_k_m",
)
!ls -lh nib-qwen-q4_k_m/

## 8. Download to local Mac

In [ ]:
import os, glob
from google.colab import files

# Unsloth's save_pretrained_gguf actually writes to <name>_gguf/<base>.<quant>.gguf,
# not into the directory we passed. Glob both candidate locations.
candidates = (
    glob.glob("/content/**/*.gguf", recursive=True)
    + glob.glob("**/*.gguf", recursive=True)
)
# Filter to real model files (skip any tiny tokenizer.gguf etc.)
candidates = sorted({p for p in candidates if os.path.getsize(p) > 100_000_000})
print("candidates:", candidates)
gguf = candidates[0]
print(f"using: {gguf}  size: {os.path.getsize(gguf) / 1e9:.2f} GB")
files.download(gguf)


## 9. Adapter-only export (v2.1)

Save the trained LoRA *without* merging back into the base, then convert it to a GGUF adapter (~36 MB f16) using `llama.cpp/convert_lora_to_gguf.py`. This is what ships in Nib v2.1 alongside the stock Qwen 2.5-1.5B base — every future Nib LoRA can ship as a tiny adapter swap instead of a 940 MB full-model re-upload.

**Skip this if you only want the v2.0-style merged GGUF.** Cells 7 + 8 above already produced that.

In [ ]:
# Save the LoRA adapter alone (no merge). PEFT writes
# adapter_model.safetensors + adapter_config.json; the base model
# stays referenced by name in the config.
ADAPTER_DIR = "nib-faithful-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}

# llama.cpp's convert_lora_to_gguf.py needs the HF base model present
# locally to map LoRA tensor names. Pull Qwen 2.5-1.5B-Instruct (~3 GB
# safetensors, cached after first download).
from huggingface_hub import snapshot_download
QWEN_BASE_HF = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct")
print("base HF dir:", QWEN_BASE_HF)

import os
LLAMA_CPP = "/content/llama.cpp"
if not os.path.exists(LLAMA_CPP):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !pip install -q -r /content/llama.cpp/requirements.txt

ADAPTER_GGUF = "nib-faithful-f16.gguf"
!python {LLAMA_CPP}/convert_lora_to_gguf.py \
    --base {QWEN_BASE_HF} \
    --outfile {ADAPTER_GGUF} \
    --outtype f16 \
    {ADAPTER_DIR}

print(f"\nadapter size: {os.path.getsize(ADAPTER_GGUF) / 1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download("nib-faithful-f16.gguf")

## Next: local eval + ship

On the Mac, drop the GGUFs into the right places:

**v2.0 path (merged, ~940 MB):**

```bash
mv ~/Downloads/qwen2.5-1.5b-instruct.Q4_K_M.gguf \
   ~/quill-research/models/nib-qwen-v2-q4_k_m.gguf
```

**v2.1 path (adapter-only, ~36 MB):**

```bash
mv ~/Downloads/nib-faithful-f16.gguf \
   ~/quill/shell/src-tauri/resources/nib-faithful-f16.gguf
```

Then run the eval, swap the GGUF in `shell/src-tauri/resources/`, build with `cargo tauri build --config tauri.conf.full.json`, and ship.